# The L8s01raw2 model — a complete account

**Alexandros Papagiannakis** · CJW7323 (*E. coli*, RplA-msfGFP / HupA-mCherry) · v6 detailed report, raw two-channel input

This notebook documents one model end to end: **L8s01raw2**, the β-VAE that takes the two
pole-oriented, mean-normalised linescans (RplA, HupA) **exactly as measured** — no symmetry
decomposition — together with the cell length and the RplA and HupA mean concentrations, and
returns an 8-dimensional latent in which cell-cycle stage, growth rate, nucleoid compaction,
nucleoid position and polysome asymmetry each occupy their own axes. It is the simplest input
design of the project, and its latent carries the growth law as an explicit axis, found without
supervision.

It differs from its sibling L8s01 (`bVAE_report_L8s01.ipynb`) in one respect only: the profiles
go in as two channels instead of four symmetry-adapted ones. Everything else — the network, the
scalars, the fixed scalar noise, β, the schedule, the split, the seed — is identical, so the two
reports can be read against each other.

Every step is described: data preparation, the network layer by layer, the loss and the one fixed
number that makes the design work, training, how the latent is read and labelled, how the
symmetry structure emerges on its own, the growth-law axes, the generative side, the atlas, the
random-forest read-outs and what they rely on, recovery of pole orientation and nucleoid position
(neither an input), and the post-processing (pseudo-time, σ-binning, cycle averaging). Code is
shown verbatim from the scripts that produced the figures, with an explanation after each block.
Numbers are held-out unless stated.

**Contents**
1. The data
2. Input representation: mean-normalised, pole-oriented, two channels
3. The network, layer by layer
4. The loss — and the one fixed number that makes the model work
5. Training
6. Reading the latent: ordering, labelling, symmetry that emerges
7. The growth law, found unsupervised
8. The generative side: reconstructions, traversals, the whole population
9. The atlas
10. The read-outs: the random forest, explained and dissected
11. Pole identity: orientation, its recovery, and how polarity is predicted
12. Nucleoid position: never an input, recovered
13. Post-processing: pseudo-time, **length vs the VAE as a cycle clock**, σ-binning, cycle averaging
14. L8s01raw2 vs L8s01: what the decomposition did and did not buy
15. Caveats, settings, files

## 1. The data

**Source.** CJW7323 (*E. coli* RplA-msfGFP / HupA-mCherry) growing in a mother-machine microfluidic
device, imaged in time-lapse; two experiments (31 May 2021, 5 June 2021). Cells were segmented and
tracked, giving complete division cycles. BioImage Archive **S-BIAD1658**.

**Unit of analysis.** One *snapshot* = one cell at one frame: **187,681** snapshots from **4,122**
complete cell cycles. Every snapshot carries:

| quantity | how it is obtained | role in L8s01raw2 |
|---|---|---|
| RplA linescan (100 bins) | fluorescence integrated across the cell width, along the long axis, mean-normalised | **input** (shape) |
| HupA linescan (100 bins) | same, second channel | **input** (shape) |
| cell length (µm) | from the segmentation mask | **input** (scalar) |
| mean RplA intensity | mean pixel intensity in the mask × per-session excitation factor (`rpla_ratio` = 1.4106 for the 1-min-interval session), log-transformed | **input** (scalar) |
| mean HupA intensity | same, `hupa_ratio` = 1.5862 | **input** (scalar) |
| cell-cycle phase | 0 at birth → 1 at division, from tracking | **held out** (target / label) |
| growth rate λ_cc | slope of ln(area) over the whole cycle, from tracking; one value per cycle, stamped on all its frames | **held out** |
| nucleoid position | signed offset of the nucleoid centroid(s) from midcell, from independent 2-D HupA segmentation | **held out** |
| nucleoid number, sister distance | from the same 2-D segmentation | label only |

"Held out" means the model never sees the quantity in training; it is used afterwards only to
evaluate and label the representation.

**The split.** 159,835 training snapshots / 27,846 held-out, split **by cell cycle** (618 cycles
entirely held out), so no frame of a held-out cycle is ever in training. Everything reported as
held-out is computed on those 27,846 frames.

![data overview](figures/figv6_2_data_overview_L8s01raw2.png)

*Top:* four held-out cells across the cycle, RplA (green) and HupA (purple) profiles along the
pole-oriented axis (new pole left). *Bottom:* distributions of the three scalar inputs and of the
log concentration ratio, training vs held-out.

## 2. Input representation

Two transformations stand between a raw snapshot and what the network sees — one fewer than in
L8s01.

**2.1 Mean-normalisation of the profiles.** Each linescan is divided by its own mean, so every
profile has mean 1. This removes absolute intensity — illumination, exposure, expression level —
and leaves *shape*: where the signal sits, how peaked the nucleoid is, how asymmetric the
polysomes are. The concentration information it removes is supplied separately as the two scalar
intensities (2.3).

**2.2 Pole orientation.** Profiles are oriented by lineage: the *new* pole (born at the last
division) on the left, the *old* pole on the right. The left and right halves are therefore not
interchangeable, and the representation can carry **signed** spatial information — which pole the
nucleoid leans to, which pole the polysomes favour. Orientation currently needs tracking; section
11 measures what happens without it and how far the model recovers it.

**No symmetry decomposition.** L8s01 split each profile into a symmetric and an antisymmetric
half so that pole-blind and pole-signed information entered on separate channels. L8s01raw2 does
not: the network receives the two centred profiles x − 1 and must discover for itself that a
mirror image is a meaningful transformation. Section 6 shows that it does — the axes come out
typed as pole-blind or pole-signed at least as cleanly as with the decomposition.

![input channels](figures/figv6_3_symanti_decomposition_L8s01raw2.png)

*Three held-out cells. Columns 1 and 4 are the measured profiles; columns 2 and 5 are the two
input channels (centred); columns 3 and 6 show what the same cell looks like after a pole flip —
the transformation the network has to learn the meaning of.*

**2.3 The scalars.** Three numbers per cell: length, log mean RplA, log mean HupA, each
excitation-corrected and then **standardised on the training split** (means 3.04 µm, 6.06, 5.31;
SDs 0.79, 0.157, 0.242). Standardising puts them on a common scale so one fixed noise level
(section 4) means the same for all three.

**What the encoder receives per cell:** a 2 × 100 array and a 3-vector. Nothing else.

### 2.4 The code

The two-channel input is the trivial case of `prep_profiles` (`vae_model_v4.py`) — subtract 1:

In [ ]:
def prep_profiles(X2, ck):
    """(N, 2, 100) mean-normalised profiles -> network input for this checkpoint:
    4 sym/anti channels (ck['channels'] == 4) or the 2 centred profiles (== 2)."""
    import numpy as np
    if ck.get("channels", 4) == 4:
        return sym_anti(X2 - 1.0)
    return (X2 - 1.0).astype(np.float32)


* For a checkpoint trained with `channels == 2`, the input is simply the mean-normalised
  profiles minus 1 (so a flat profile is all zeros); for the 4-channel models the same function
  applies the sym/anti split. The training script does the same thing inline (`raw2=True`).

The scalar inputs (`train_vae_v6.py`):

In [ ]:
def scalars_v6(d):
    f = d["features"]
    fn = [str(x) for x in d["feature_names"]]
    is1m = np.array(["1minint" in str(t) for t in d["traj_ids"]])[d["traj_idx"]]
    log_r = np.log(f[:, fn.index("mean_fluor_2")] * np.where(is1m, 1.4106, 1.0))
    log_h = np.log(f[:, fn.index("mean_fluor_3")] * np.where(is1m, 1.5862, 1.0))
    return np.c_[d["length"], log_r, log_h]


* `is1m` flags the frames from the 1-minute-interval session, whose excitation differed; the
  per-session factors (`rpla_ratio` 1.4106, `hupa_ratio` 1.5862) put both sessions on one
  intensity scale before the log.
* Returns `(N, 3)`: length, log corrected mean RplA, log corrected mean HupA. Standardisation uses
  training-split statistics only.

## 3. The network, layer by layer

The architecture is the v2 network with 2 input channels and 3 scalars; **964,826** parameters
(the only difference from L8s01 is the first convolution's input width and the last
deconvolution's output width).

**Encoder q_φ(z | x, s)** — from the 2 × 100 profiles and the 3 scalars to a distribution over z:

| layer | operation | output shape | role |
|---|---|---|---|
| conv1 | Conv1d(2 → 32, kernel 5, stride 2, pad 2) + GELU | 32 × 50 | local shape features, both colours mixed |
| conv2 | Conv1d(32 → 64, k 5, s 2, p 2) + GELU | 64 × 25 | mid-scale features (peaks, gaps) |
| conv3 | Conv1d(64 → 128, k 5, s 2, p 2) + GELU | 128 × 13 | coarse features (halves of the cell) |
| flatten + concat | 128·13 = 1664 features ‖ 3 scalars | 1667 | **the scalars enter here** |
| fc | Linear(1667 → 256) + GELU | 256 | joint representation of shape and scalars |
| μ, log σ² | two Linear(256 → 8) | 8 + 8 | the posterior mean and log-variance of z |

Convolutions are translation-equivariant but not mirror-equivariant: a flipped profile produces
flipped feature maps, and whether the fully-connected layer treats the two as "the same cell, other
way round" or as different cells is entirely learned. That is what makes the emergence of
pole-signed axes in section 6 a result rather than a construction.

**Sampling.** z = μ + σ ⊙ ε with ε ~ N(0, I) during training; at analysis time μ is the cell's
coordinate and σ its uncertainty.

**Decoder p_θ(x, s | z)** — from 8 numbers back to 2 × 100 profiles and 3 scalars:

| layer | operation | output shape |
|---|---|---|
| fc | Linear(8 → 256) + GELU, Linear(256 → 1664) + GELU, reshape | 128 × 13 |
| deconv1–3 | ConvTranspose1d 128 → 64 → 32 → **2**, kernel 5, stride 2 (lengths 13 → 25 → 50 → 100) | 2 × 100 |
| scalar head | Linear(8 → 32) + GELU, Linear(32 → 3) | 3 |

The scalar head is a separate branch from z: the decoder must reproduce length and the two
concentrations *from the latent alone*. That is the only route by which the scalars can be pulled
into z — and why section 4 matters.

**Observation noise.** Each of the 2 profile channels has a learned σ_x; each of the 3 scalars has
a noise σ_s that is **fixed at 0.1**.

![schematic](figures/figv6_0_schematic_L8s01raw2.png)

The schematic follows one held-out cell (phase 0.60, 3.7 µm); the latent circles are coloured by
the cell's μ and labelled by each axis's strongest correlate (section 6).

### 3.1 The code — the network (`vae_model_v4.py`)

In [ ]:
class VAE(nn.Module):
    """zdim latent dims, `channels` linescan channels, `n_scalars` scalar inputs."""

    def __init__(self, zdim=8, channels=4, n_scalars=1):
        super().__init__()
        self.channels = channels
        self.n_scalars = n_scalars
        self.enc = nn.Sequential(
            nn.Conv1d(channels, 32, 5, stride=2, padding=2), nn.GELU(),
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.GELU(),
            nn.Conv1d(64, 128, 5, stride=2, padding=2), nn.GELU(),
            nn.Flatten(),
        )
        self.enc_fc = nn.Sequential(nn.Linear(128 * 13 + n_scalars, 256), nn.GELU())
        self.mu = nn.Linear(256, zdim)
        self.logvar = nn.Linear(256, zdim)
        self.dec_fc = nn.Sequential(
            nn.Linear(zdim, 256), nn.GELU(), nn.Linear(256, 128 * 13), nn.GELU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose1d(128, 64, 5, stride=2, padding=2, output_padding=0),
            nn.GELU(),
            nn.ConvTranspose1d(64, 32, 5, stride=2, padding=2, output_padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(32, channels, 5, stride=2, padding=2, output_padding=1),
        )
        self.dec_len = nn.Sequential(nn.Linear(zdim, 32), nn.GELU(),
                                     nn.Linear(32, n_scalars))
        self.log_sig2_x = nn.Parameter(torch.zeros(channels))
        self.log_sig2_l = nn.Parameter(torch.zeros(n_scalars))

    def encode(self, x, s):
        h = self.enc(x)
        if s.dim() == 1:
            s = s[:, None]
        h = self.enc_fc(torch.cat([h, s], dim=1))
        return self.mu(h), self.logvar(h)

    def decode(self, z):
        h = self.dec_fc(z).view(-1, 128, 13)
        shat = self.dec_len(z)
        return self.dec(h)[:, :, :100], (shat[:, 0] if self.n_scalars == 1 else shat)

    def forward(self, x, s):
        mu, logvar = self.encode(x, s)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        xhat, shat = self.decode(z)
        return xhat, shat, mu, logvar


* `__init__`: `enc` is the three strided convolutions (100 → 50 → 25 → 13) and `Flatten`;
  `enc_fc` takes 128·13 = 1664 features **plus `n_scalars`** to 256 units; `mu` and `logvar` are
  the heads. `dec_fc` inverts the path to 128 × 13; `dec` is three transposed convolutions back to
  `channels` × 100 (cropped `[:, :, :100]`); `dec_len` is the scalar head. `log_sig2_x` and
  `log_sig2_l` are the per-channel and per-scalar log observation variances — learned unless frozen.
* `encode(x, s)`: convolutions, then `torch.cat([h, s], dim=1)` appends the scalars to the
  flattened features; returns μ and log σ².
* `decode(z)`: profiles and scalars from z.
* `forward`: encode, sample `mu + randn * exp(0.5 * logvar)`, decode.

In [ ]:
import sys, torch, numpy as np
sys.path.insert(0, "../v4")
from vae_model_v4 import load_model
model, ck = load_model("vae_L8s01raw2.pt")
print(model)
print("channels:", ck["channels"], "| fixed scalar sigma:", np.round(model.log_sig2_l.exp().sqrt().detach().numpy(), 3),
      "| learned profile sigma:", np.round(model.log_sig2_x.exp().sqrt().detach().numpy(), 4))
print("parameters:", sum(p.numel() for p in model.parameters()))

## 4. The loss — and the one fixed number that makes the model work

The model minimises, per cell,

$$\mathcal{L} = \underbrace{\sum_{c=1}^{2}\sum_{i=1}^{100} \tfrac12\Big[\tfrac{(x_{ci}-\hat x_{ci})^2}{\sigma_{x,c}^2} + \log\sigma_{x,c}^2\Big]}_{\text{profile reconstruction}}
\;+\; \underbrace{\sum_{k=1}^{3} \tfrac12\Big[\tfrac{(s_k-\hat s_k)^2}{\sigma_{s}^2} + \log\sigma_{s}^2\Big]}_{\text{scalar reconstruction}}
\;+\; \beta\,\underbrace{\mathrm{KL}\big(q_\phi(z|x,s)\,\|\,\mathcal N(0,I)\big)}_{\text{capacity penalty}},\qquad \beta = 2.$$

Each reconstruction term is a Gaussian negative log-likelihood: an error is charged in units of the
assumed noise. The KL term charges for every nat of information the latent carries; β = 2 doubles
that charge, pushing the model toward few, well-separated axes.

**Why the scalars were lost before (v3a, v4-L8c, v5-L8r).** With σ_s *learned*, the cheapest way
to handle 3 scalars competing with 200 profile values is to **raise σ_s** — declare them noisy —
after which their error is nearly free and the KL term reclaims the capacity. σ_s ended near
0.8 SD every time, and the latent kept only the ≈ 40 % of the concentrations that shape implies.

**The v6 change.** σ_s is fixed at **0.1 SD** and removed from the optimiser. Physically: length
and the two mean intensities are measured to roughly a tenth of their population spread, and the
model may not argue otherwise. An error of 0.8 SD on a concentration now costs (0.8/0.1)² / 2 =
32 nats — far more than the ≈ 2–3 nats a latent dimension costs under β = 2 — so encoding the
scalars is the cheaper option. Nothing else changes.

### 4.1 The code — the loss (`vae_model_v4.py`)

In [ ]:
def elbo_terms(m, x, s, xhat, shat, mu, logvar):
    """Gaussian NLL with per-channel and per-scalar learned variance, plus KL."""
    v2 = m.log_sig2_x.exp()[None, :, None]                  # (1, C, 1)
    nll_x = (0.5 * ((x - xhat) ** 2 / v2
                    + m.log_sig2_x[None, :, None])).sum((1, 2)).mean()
    if s.dim() == 1:
        s = s[:, None]
    if shat.dim() == 1:
        shat = shat[:, None]
    nll_s = (0.5 * ((s - shat) ** 2 / m.log_sig2_l.exp()[None, :]
                    + m.log_sig2_l[None, :])).sum(1).mean()
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(1).mean()
    return nll_x, nll_s, kl


* `v2 = exp(log_sig2_x)` broadcast over `(batch, channel, position)`; `nll_x` is the Gaussian
  NLL of the profiles — squared error over the channel's variance plus the `log σ²` term that stops
  σ from shrinking to zero for free — summed over channels and positions, averaged over the batch.
* `nll_s`: the same for the scalars with `log_sig2_l`.
* `kl`: closed-form KL between the diagonal Gaussian posterior and N(0, I).
* Training loss: `nll_x + nll_s + BETA * kl`.

The lines that fix the scalar noise and select the raw input (`train_vae_v6.py`):

In [ ]:
    X4 = ((d["X"][:, [0, 1]] - 1.0).astype(np.float32) if raw2
          else sym_anti(d["X"][:, [0, 1]] - 1.0))
    n_ch = X4.shape[1]


In [ ]:
    model = VAE(ZDIM, channels=n_ch, n_scalars=3).to(dev)
    # fixed, known scalar observation noise (not learned)
    model.log_sig2_l.data.fill_(float(np.log(sig_s ** 2)))
    model.log_sig2_l.requires_grad_(False)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)


* With `raw2=True` the input is the centred 2-channel array; `n_ch` sets the network width.
* `log_sig2_l.data.fill_(log(σ_s²))` sets the scalar variance (σ_s = 0.1 → log σ² = −4.6);
  `requires_grad_(False)` freezes it; the optimiser is built only from the parameters that still
  require gradients.

## 5. Training

Adam, learning rate 10⁻³, batch 512, **40 epochs** over 159,835 training snapshots, seed 0, on
Apple MPS — the schedule of every version since v2.

![training](figures/figv6_4_training_L8s01raw2.png)

*Left:* validation −ELBO. Its absolute value is not comparable with the 4-channel models (half as
many profile values are scored), so read the shape: steep descent as the profile noise shrinks,
plateau after ≈ 20 epochs. *Middle:* KL — the information in the latent — at epoch 1 is already
7.85 nats (the scalars are pulled in immediately; learned-σ models start at 0.6) and ends at
**17.86 nats/cell**. *Right:* the scalar reconstruction error on held-out cells ends at
**0.262 / 0.228 / 0.220 SD** for length / log RplA / log HupA — tighter than L8s01's
0.32 / 0.40 / 0.39. With only two profile channels to reconstruct, the scalars compete against
fewer values and the fixed σ_s holds them more firmly.

After training, every snapshot is encoded once; μ ∈ ℝ⁸ and σ ∈ ℝ⁸ are saved in
`latents_L8s01raw2.npz` and are what all later analysis uses.

### 5.1 The code — the training loop (`train_vae_v6.py`)

In [ ]:
    for ep in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, BATCH):
            idx = perm[i: i + BATCH]
            x, l = Xtr[idx].to(dev), Ltr[idx].to(dev)
            xhat, lhat, mu, logvar = model(x, l)
            nll_x, nll_l, kl = elbo_terms(model, x, l, xhat, lhat, mu, logvar)
            loss = nll_x + nll_l + BETA * kl
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                xhat, lhat, mu, logvar = model(Xva, Lva)
                nll_x, nll_l, kl = elbo_terms(model, Xva, Lva, xhat, lhat, mu, logvar)
                rmse = ((lhat - Lva) ** 2).mean(0).sqrt()
            print(f"[{tag}] ep {ep:3d} val_elbo {(nll_x + nll_l + BETA * kl).item():9.2f} "
                  f"KL {kl.item():6.2f} scalar RMSE (SD units) " +
                  " ".join(f"{v:.3f}" for v in rmse.tolist()), flush=True)


* Each epoch visits every training cell once in random order (`torch.randperm`), in batches of
  512; `loss = nll_x + nll_l + BETA * kl` is minimised with Adam.
* Every tenth epoch the held-out set is pushed through and −ELBO, KL and the scalar RMSE (SD units)
  are logged — the curves above.

The encoding pass that produces the latents:

In [ ]:
    model.eval()
    mus, sds = [], []
    with torch.no_grad():
        Xall, Lall = torch.from_numpy(X4), torch.from_numpy(S)
        for i in range(0, len(Xall), 4096):
            mu, logvar = model.encode(Xall[i: i + 4096].to(dev), Lall[i: i + 4096].to(dev))
            mus.append(mu.cpu().numpy()); sds.append(np.exp(0.5 * logvar.cpu().numpy()))
    Z, SIG = np.concatenate(mus), np.concatenate(sds)


* All 187,681 cells in chunks of 4,096; `encode()` returns μ and log σ²; σ = exp(½ log σ²).
* `Z` are the coordinates used everywhere below; `SIG` sets the σ-binning resolution (section 13).

## 6. Reading the latent: ordering, labelling, and symmetry that emerges

**Ordering.** The eight axes are sorted by the SD of their posterior means (largest first) and
named z1 … z8. The SDs are all ≈ 1 (β-VAE latents are pulled toward unit variance), so this is a
labelling, not a ranking.

**Labelling.** Each axis is labelled by its strongest Spearman correlate among: cycle, nucleoid
position, sister-nucleoid distance, compaction, nucleoid pole asymmetry, polysome displacement,
midcell polysome — except the two axes that move the concentrations most (section 7), which are
labelled by *which* concentration combination they move.

**Symmetry type — now a measurement, not a design.** In L8s01 the input channels were
symmetry-typed, so an axis could be classified by which channels it drove. Here the inputs are not
typed, so the symmetry of an axis is measured on its *decoded response*: each axis is perturbed by
±0.1, the decoded 2 × 100 response J is split into its even and odd halves about midcell,
J_anti = ½(J − flip J), and the **antisymmetric share** ‖J_anti‖² / ‖J‖² is 0 for a pole-blind axis
and 1 for a pole-signed one. Independently, the *encoder's* equivariance is measured by encoding
every cell and its flipped copy and correlating the two coordinates axis by axis: +1 = invariant,
−1 = sign-reversing.

![latent overview](figures/figv6_5_latent_overview_L8s01raw2.png)

![correlations](figures/figv6_1_correlations_L8s01raw2.png)

| axis | label (strongest correlate) | antisymmetric share (decoder) | corr(z, z_flip) (encoder) |
|---|---|---|---|
| z1 | sister-nucl. distance | 0.07 | 0.966 |
| z2 | nucleoid position | 0.88 | -0.665 |
| z3 | nucl. pole asym. | 0.96 | -0.735 |
| z4 | compaction | 0.18 | 0.521 |
| z5 | log HupA axis | 0.2 | 0.942 |
| z6 | log RplA axis | 0.45 | 0.933 |
| z7 | midcell polysome | 0.11 | 0.863 |
| z8 | polysome displ. | 0.84 | -0.607 |

**The symmetry structure emerged on its own — and cleanly.** Three axes are pole-signed
(antisymmetric share ≥ 0.83, flip correlation −0.61 … −0.73): z2 (nucleoid position), z3 (nucleoid
pole asymmetry) and z8 (polysome displacement). Four are pole-blind (flip correlation +0.86 …
+0.97): z1 (cycle/size), z7 (midcell polysome), and z5 and z6 — the concentration axes, which a
flip cannot touch; their profile response is so small (section 8) that their antisymmetric share
(0.20, 0.45) is a ratio of two near-zero numbers and carries no meaning. z4 (compaction) is the one
mixed axis (share 0.18, flip +0.52). Compared with L8s01,
whose signed axes reversed at only −0.34 … −0.69, the raw-input model's signed axes are *more*
equivariant: the decomposition was a device for making this structure explicit, and the network
finds it unaided — which is what v4 had already observed about v3's axes (most were typed; only the
polysome sector was mixed).

### 6.1 The code — ordering, labelling, symmetry (`figures_L8s01.py`, `inspect_v6.py`)

In [ ]:
n = np.load(f"{HERE}/latents_{TAG}.npz")
order = np.argsort(-n["Z"].std(0)); inv = np.argsort(order)
Z, SIG = n["Z"][:, order], n["SIG"][:, order]


* `order = argsort(−SD)` sorts the axes; `inv` is the inverse permutation, used whenever a
  sorted-space vector is sent back into the decoder (which expects the trained order).

In [ ]:
anti = []
for j in range(zdim):
    zp = np.zeros((2, zdim), np.float32); zp[0, j], zp[1, j] = 0.1, -0.1
    with torch.no_grad():
        x, _ = model.decode(torch.from_numpy(zp[:, inv]))
    J = reassemble((x[0:1] - x[1:2]).numpy(), ck)[0] - 1.0
    Ja = 0.5 * (J - J[:, ::-1])
    anti.append((Ja ** 2).sum() / (J ** 2).sum())
# decoder response of each axis on the scalar head (which axes carry the scalars?)
with torch.no_grad():
    zp = np.zeros((2 * zdim, zdim), np.float32)
    for j in range(zdim):
        zp[2 * j, j], zp[2 * j + 1, j] = 1.0, -1.0
    _, sh = model.decode(torch.from_numpy(zp[:, inv]))
sh = sh.numpy().reshape(zdim, 2, -1)
scalar_resp = (sh[:, 0] - sh[:, 1]) / 2                   # (zdim, 3) d scalar / d z
Xc = X[:, [0, 1]].astype(np.float32)
mus = []
with torch.no_grad():
    X4f = prep_profiles(Xc[:, :, ::-1].copy(), ck)
    for i in range(0, len(X4f), 4096):
        mu, _ = model.encode(torch.from_numpy(X4f[i:i + 4096]), torch.from_numpy(S[i:i + 4096]))
        mus.append(mu.numpy())
Zf = np.concatenate(mus)[:, order]
equi = [np.corrcoef(Z[:, j], Zf[:, j])[0, 1] for j in range(zdim)]


* For each axis a pair of latent points (±0.1 on that axis) is decoded; `reassemble` brings the
  response to measured space (for a 2-channel model, identity); `Ja = ½(J − J[:, ::-1])` is its
  odd part; `anti` is the odd share of the energy.
* The scalar-head block decodes ±1 on each axis and reads the scalars: `scalar_resp[j]` =
  ∂[length, log RplA, log HupA] / ∂z_j (panel A of section 7).
* The equivariance block encodes every cell's flipped profile (`prep_profiles(Xc[:, :, ::-1])`)
  with the same scalars and correlates with the original coordinates.

## 7. The growth law, found unsupervised

The growth law states that the ribosome fraction of the proteome rises linearly with growth rate.
Its per-cell proxy here is the **RplA/HupA mean-intensity ratio** — ribosomes per unit DNA —
which correlates with the cycle growth rate at Spearman +0.63, while either intensity alone does
not (+0.30 / −0.54). The two log intensities are given to the model as two separate numbers.

![growth law](figures/figv6_6_growth_law_L8s01raw2.png)

**A — what each axis does to the scalars.** Differentiating the decoder's scalar head along each
latent axis gives the matrix shown. Two axes move the concentrations: **z5**, with response
(log RplA +0.00, log HupA -0.93) — log HupA — and
**z6**, with (+0.88, +0.13) — log RplA. The two
concentration scalars span a two-dimensional plane in the latent, and this model's basis for that
plane is **one axis per channel** rather than L8s01's ratio/sum pair. Both are legitimate
coordinates of the same plane: the β-VAE objective is nearly indifferent to rotations within a
plane of equal-variance axes, so *which* basis appears is seed-dependent; *that* the plane is
spanned by two dedicated axes is the design's result.

**B, C — confirmation against the inputs.** Each concentration axis plotted against the
input-derived combination it encodes (its own response direction): ρ = +0.91 for z5,
+0.86 for z6.

**D — where growth sits.** Binning cells by each axis and averaging the tracked growth rate:
**z5 (log HupA) carries growth at ρ = +0.52** — the strongest single-axis
growth correlation of the project (L8s01's ratio axis: −0.45; every earlier model: ≤ 0.32 via the
cycle axis). z6 (log RplA) carries +0.15; z1 (cycle/size) carries
+0.35, the growth law's size dependence.

**E, F — the concentration plane.** Cells in (z5, z6) coloured by growth rate, with the
input ratio and brightness directions drawn as arrows: growth varies along the ratio arrow, which
runs diagonally through this basis. Coloured by phase: no structure — the cycle lives elsewhere.

**Reading the biology.** An axis that is (to first order) −log HupA reads as "DNA concentration,
inverted": at a given ribosome content, cells with less DNA per volume are the faster growers.
That is the growth law seen from the DNA side — the ratio's denominator — and it is the same
statement as the ribosome-side version in L8s01 expressed in a rotated basis. The read-out
(section 10) combines the two axes and recovers the ratio itself at R² 0.941 from the latent
alone.

Decoder response of each axis on the scalar head, d[length, log RplA, log HupA] / dz:

```
   z1: +0.84 +0.46 -0.22
   z2: +0.33 +0.02 -0.14
   z3: +0.00 -0.06 +0.08
   z4: +0.31 +0.10 -0.03
   z5: +0.00 +0.00 -0.93
   z6: -0.13 +0.88 +0.13
   z7: -0.28 -0.05 +0.03
   z8: +0.02 +0.04 +0.03
```

## 8. The generative side

Any point z decodes to a cell. For this model the decoded channels *are* the measured profiles
(plus 1), so no reassembly is needed.

**Reconstructions** — six held-out cells across the cycle, measured (solid) vs decoded from their
own μ (dashed): nucleoid splitting, polysome redistribution and the asymmetries are kept; pixel
noise is smoothed.

![reconstructions](figures/figv4_3_reconstructions_L8s01raw2.png)

**Traversals in measured space** — one axis at a time, −2 … +2 SD, other axes at 0, each column
labelled by correlate and by its *measured* symmetry type.

![traversals](figures/figv4_4_traversals_L8s01raw2.png)

**Traversals in the input channels**, with the scalar head's change over each sweep in the title:
the shape axes move the profiles and leave the scalars nearly alone; **z5 and z6 move the
scalars and leave the profiles almost flat** — the concentration axes are orthogonal to shape in
the generator.

![traversals input channels](figures/figv6_8_traversals_symanti_L8s01raw2.png)

**The whole population** — all 187,681 linescans as rows, sorted four ways: by z1 the
nucleoid-splitting programme unfolds as by true phase; by z5 (a concentration axis) the profiles
show no order — it is not a shape axis.

![population linescans](figures/figv6_7_population_linescans_L8s01raw2.png)

## 9. The atlas

A t-SNE of the 8-dimensional latent (20,000 cells, perplexity 50) — a map for looking, not for
measuring.

![atlas](figures/figv4_5_atlas_L8s01raw2.png)

Phase runs across the map as a continuous progression; position and polysome asymmetry split it
into signed halves; growth forms its own gradient because the concentration axes are part of the
embedded coordinates.

**σ-binning on the atlas** — each axis cut at −1σ / 0 / +1σ, with the occupancy of the σ-resolved
identities (section 13):

![sigma binning](figures/figv4_7_sigma_binning_L8s01raw2.png)

## 10. The read-outs: the random forest, explained and dissected

The latent is unsupervised: no axis was told what to be. To turn coordinates into *calibrated*
quantities — a phase in [0, 1], a growth rate in h⁻¹, a position in cell lengths — a supervised
**read-out** is fitted afterwards on the training cycles, where tracking gives the true values, and
evaluated on the held-out cycles, which it treats as snapshots.

### 10.1 What a random forest is, and how it makes a prediction

**100 decision trees**, each grown on a bootstrap sample of the 159,835 training frames. A tree is
built by repeatedly choosing, among a random subset of the inputs, the feature and threshold that
best split the cells into two groups with different target values, until a leaf holds **≥ 5
cells**. A new cell is dropped through every tree, lands in one leaf per tree, and its prediction
is the **average of the training targets in those 100 leaves**. A data-adaptive lookup table: bins
chosen by the data, different in every tree, averaged. No feature scaling, interactions found on
its own, no extrapolation beyond the training range. Settings:
`RandomForestRegressor(100, min_samples_leaf=5, random_state=0)`; no tuning.

**Inputs to the read-out: the eight latent means μ and the cell length.** The concentrations are
not given to it — they are inside μ (giving them anyway changes nothing: read-outs A and C below).

### 10.2 The code — read-out functions (`evaluate_v6.py`)

In [ ]:
def r2(zm, target, linear=False):
    ok = np.isfinite(target)
    m = (LinearRegression() if linear else
         RandomForestRegressor(100, n_jobs=-1, random_state=0, min_samples_leaf=5))
    m.fit(zm[tr & ok], target[tr & ok])
    return r2_score(target[va & ok], m.predict(zm[va & ok]))


ntraj = traj.max() + 1
gr_t = np.zeros(ntraj); val_t = np.zeros(ntraj, bool)
for i in range(len(traj)):
    gr_t[traj[i]] = cc_gr[i]; val_t[traj[i]] = is_val[i]


def r2_cycle(zm):
    Zt = np.zeros((ntraj, zm.shape[1])); np.add.at(Zt, traj, zm)
    Zt /= np.bincount(traj, minlength=ntraj)[:, None]
    ok = np.isfinite(gr_t)
    rf = RandomForestRegressor(200, n_jobs=-1, random_state=0)
    rf.fit(Zt[ok & ~val_t], gr_t[ok & ~val_t])
    return r2_score(gr_t[ok & val_t], rf.predict(Zt[ok & val_t]))


* `r2(zm, target)`: fit on training rows, score on held-out rows; `zm` is whatever the read-out
  is given, so one function serves every variant; `linear=True` swaps in a least-squares line.
* `r2_cycle(zm)`: one row per cycle (features averaged over the cycle's frames with `np.add.at`),
  target = the cycle's growth rate, split by cycle, 200 trees.

In [ ]:
feat = np.c_[Z, length]; names = [f"z{j+1}" for j in range(zdim)] + ["length"]
TARG = [("cell-cycle phase", phase, "twilight"), ("growth rate λ_cc (h⁻¹)", cc_gr, "cividis"),
        ("nucleoid position", med_pos, "coolwarm"), ("nucleoid compaction", nuc.var(1), "viridis")]
fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 4)
imp = {}
preds = {}
for k, (nm, y, cm) in enumerate(TARG):
    okk = np.isfinite(y)
    rf = RandomForestRegressor(100, n_jobs=-1, random_state=0, min_samples_leaf=5).fit(feat[tr & okk], y[tr & okk])
    p = rf.predict(feat[va & okk]); preds[nm] = (y[va & okk], p)
    s_ = rng.choice(np.where(va & okk)[0], 6000, replace=False)
    pi = permutation_importance(rf, feat[s_], y[s_], n_repeats=3, random_state=0, n_jobs=-1)
    imp[nm] = pi.importances_mean


* `feat` = the eight sorted latent means + length. For each target a forest is fitted on the
  training rows and `permutation_importance` shuffles one column of a 6,000-cell held-out sample
  at a time (3 repeats), recording the drop in R².

In [ ]:
base = np.median(feat[tr], 0)
for j, c, nm in ((G, C_Z7, f"z{G+1} ({conc_label(G)})"), (0, "#888", "z1 (cycle)"), (O, C_Z8, f"z{O+1} ({conc_label(O)})")):
    grid = np.linspace(*np.percentile(Z[:, j], [3, 97]), 25)
    Xg = np.tile(base, (25, 1)); Xg[:, j] = grid
    ax.plot(grid, rf.predict(Xg), color=c, lw=1.6, label=nm)


* Partial dependence: every feature at its training median except one, swept over its 3rd–97th
  percentile; the forest's prediction along the sweep is the curve.

### 10.3 What the forest relies on — permutation importance

| target | z1 | z2 | z3 | z4 | z5 | z6 | z7 | z8 | length |
|---|---|---|---|---|---|---|---|---|---|
| cell-cycle phase | 0.07 | 0.02 | 0.01 | 0.01 | 0.27 | 0.12 | 0.00 | 0.00 | 0.94 |
| growth rate λ_cc (h⁻¹) | 0.20 | 0.03 | 0.02 | 0.06 | 0.56 | 0.07 | 0.02 | 0.01 | 0.03 |
| nucleoid position | 0.39 | 0.79 | 0.03 | 0.11 | 0.01 | 0.01 | 0.01 | 0.05 | 0.02 |
| nucleoid compaction | 0.50 | 0.43 | 0.01 | 0.74 | 0.00 | 0.04 | 0.09 | 0.00 | 0.00 |

* **Phase** is read from length (0.94), z5 (0.27), z6 (0.12).
* **Growth** is read from z5 (0.56), z1 (0.20), z6 (0.07) — the growth law read straight off the concentration
  axis.
* **Nucleoid position** from z2 (0.79), z1 (0.39), z4 (0.11).
* **Compaction** from z4 (0.74), z1 (0.50), z2 (0.43).

![read-out](figures/figv6_9_readout_rf_L8s01raw2.png)

### 10.4 The shape of the growth read-out — partial dependence

Sweeping z5 with everything else at its median, the predicted growth rate moves monotonically
across ≈ 0.2 h⁻¹ — a near-linear response that *is* the growth law; along z6 and z1 it moves
far less.

### 10.5 Held-out accuracy

| held-out target | R² (latent + length) |
|---|---|
| cell-cycle phase | **0.823** |
| growth rate, per frame | **0.514** |
| growth rate, cycle-averaged (618 cycles) | **0.580** |
| nucleoid position | 0.699 |
| nucleoid compaction | 0.951 |

Growth is the noisiest target because λ_cc is one number per cycle stamped on every frame;
averaging a cell's frames before predicting lifts it to 0.58.

### 10.6 The full read-out comparison (`evaluate_v6.py`)

Z = latent only; B = latent + length; A = the v3/v4 standard (+ both log concentrations); C =
+ log ratio. L8 and L8c (v4, length-only / learned-σ) and L8s01 (sym/anti) for comparison.

**Z — latent only**

| model | phase | growth (frame) | growth (cycle-avg) | position | compaction | polysome asym. |
|---|---|---|---|---|---|---|
| L8 (length only) | 0.701 | 0.373 | 0.472 | 0.714 | 0.960 | 0.880 |
| L8c (+ conc., learned sigma_s) | 0.704 | 0.387 | 0.474 | 0.715 | 0.961 | 0.885 |
| L8s01 (+ conc., sigma_s = 0.1) — v6 | 0.778 | 0.491 | 0.559 | 0.695 | 0.949 | 0.831 |
| L8s03 (+ conc., sigma_s = 0.3) — v6 | 0.710 | 0.403 | 0.507 | 0.712 | 0.959 | 0.859 |
| L8s01raw2 (raw 2-ch linescans, sigma_s = 0.1) — v6 | 0.793 | 0.513 | 0.550 | 0.704 | 0.952 | 0.792 |

**B — latent + length** (the read-out used in this notebook)

| model | phase | growth (frame) | growth (cycle-avg) | position | compaction | polysome asym. |
|---|---|---|---|---|---|---|
| L8 (length only) | 0.743 | 0.402 | 0.521 | 0.715 | 0.962 | 0.881 |
| L8c (+ conc., learned sigma_s) | 0.747 | 0.415 | 0.541 | 0.716 | 0.963 | 0.886 |
| L8s01 (+ conc., sigma_s = 0.1) — v6 | 0.821 | 0.499 | 0.575 | 0.691 | 0.949 | 0.831 |
| L8s03 (+ conc., sigma_s = 0.3) — v6 | 0.764 | 0.433 | 0.552 | 0.712 | 0.961 | 0.864 |
| L8s01raw2 (raw 2-ch linescans, sigma_s = 0.1) — v6 | 0.823 | 0.514 | 0.580 | 0.699 | 0.951 | 0.794 |

**A — latent + length + both concentrations**

| model | phase | growth (frame) | growth (cycle-avg) | position | compaction | polysome asym. |
|---|---|---|---|---|---|---|
| L8 (length only) | 0.818 | 0.494 | 0.573 | 0.714 | 0.961 | 0.877 |
| L8c (+ conc., learned sigma_s) | 0.818 | 0.497 | 0.583 | 0.716 | 0.962 | 0.883 |
| L8s01 (+ conc., sigma_s = 0.1) — v6 | 0.829 | 0.499 | 0.579 | 0.691 | 0.947 | 0.836 |
| L8s03 (+ conc., sigma_s = 0.3) — v6 | 0.824 | 0.500 | 0.594 | 0.711 | 0.960 | 0.857 |
| L8s01raw2 (raw 2-ch linescans, sigma_s = 0.1) — v6 | 0.822 | 0.499 | 0.577 | 0.698 | 0.949 | 0.796 |

**C — latent + length + log ratio**

| model | phase | growth (frame) | growth (cycle-avg) | position | compaction | polysome asym. |
|---|---|---|---|---|---|---|
| L8 (length only) | 0.820 | 0.508 | 0.569 | 0.714 | 0.961 | 0.880 |
| L8c (+ conc., learned sigma_s) | 0.820 | 0.509 | 0.579 | 0.715 | 0.962 | 0.884 |
| L8s01 (+ conc., sigma_s = 0.1) — v6 | 0.829 | 0.503 | 0.570 | 0.691 | 0.949 | 0.831 |
| L8s03 (+ conc., sigma_s = 0.3) — v6 | 0.825 | 0.517 | 0.574 | 0.711 | 0.961 | 0.862 |
| L8s01raw2 (raw 2-ch linescans, sigma_s = 0.1) — v6 | 0.824 | 0.501 | 0.581 | 0.698 | 0.951 | 0.793 |

For L8s01raw2 the read-outs B, A and C coincide: the read-out gains nothing from the concentrations
because the latent already holds them. Against L8s01, the raw model is equal or better on every
target but polysome asymmetry (0.792 vs 0.831).

**Linear probes** — a straight line on the latent alone:

| model | log ratio from latent (RF) | phase (linear) | growth (linear) | position (linear) | compaction (linear) |
|---|---|---|---|---|---|
| L8 (length only) | 0.397 | 0.553 | 0.168 | 0.603 | 0.628 |
| L8c (+ conc., learned sigma_s) | 0.410 | 0.567 | 0.209 | 0.616 | 0.649 |
| L8s01 (+ conc., sigma_s = 0.1) — v6 | 0.827 | 0.713 | 0.446 | 0.596 | 0.724 |
| L8s03 (+ conc., sigma_s = 0.3) — v6 | 0.437 | 0.598 | 0.275 | 0.600 | 0.671 |
| L8s01raw2 (raw 2-ch linescans, sigma_s = 0.1) — v6 | 0.941 | 0.743 | 0.465 | 0.594 | 0.740 |

The log ratio is recoverable from the raw model's latent at R² **0.941** (L8s01: 0.827; every
learned-σ model: ≈ 0.40), and a *linear* read-out of growth reaches 0.465.

## 11. Pole identity: orientation and its recovery

The profiles are pole-oriented by lineage (section 2.2). That orientation gives the pole-signed
axes their sign, and it is the one piece of tracking information still in the inputs. What does
the model lose without it, and can it recover it?

**11.1 Removing the orientation.** Every cell's profile is kept or flipped at random before
encoding (the scalars are unchanged); the labels stay pole-oriented.

![unoriented](figures/figv4_8_unoriented_correlations_L8s01raw2.png)

The pole-blind correlations survive (phase, length, compaction, nucleoid number, and the
concentration axes, which a flip cannot touch); the pole-signed columns are wiped out — nucleoid
position's correlation on z2 falls to ≈ 0. The right panel is the equivariance test —
0.966, -0.665, -0.735, 0.521, 0.942, 0.933, 0.863, -0.607 — the signed axes reverse, not perfectly, because nothing constrains the encoder
to be exactly odd under a flip; but more nearly than in L8s01.

**11.2 Recovering the orientation from the latent.** A random-forest *classifier* (200 trees) is
trained on the latents of randomly flipped training cells to predict whether each was flipped, and
tested on held-out cells.

The code (`figures_L8s01.py`):

In [ ]:
Xc = X[:, [0, 1]].astype(np.float32)
flip = rng.random(N) < 0.5
Xr = Xc.copy(); Xr[flip] = Xr[flip][:, :, ::-1]
Z_r = encode(Xr, S)                       # latent of cells whose orientation is unknown
Z_f = encode(Xc[:, :, ::-1].copy(), S)    # every cell flipped
equi = np.array([np.corrcoef(Z[:, j], Z_f[:, j])[0, 1] for j in range(zdim)])
clf = RandomForestClassifier(200, n_jobs=-1, random_state=0, min_samples_leaf=5).fit(Z_r[tr], flip[tr])
acc = clf.score(Z_r[va], flip[va])
proba = clf.predict_proba(Z_r[va])[:, 1]


* `flip` is a random half of the cells; `Xr` is the unoriented dataset; `Z_r` its latent (same
  scalars — a flip changes none). `Z_f` encodes every cell flipped; `equi` is the per-axis
  correlation with the original latent. A `RandomForestClassifier` learns `flip` from `Z_r` on
  training cells and is scored on held-out cells.

![pole recovery](figures/figv6_10_pole_recovery_L8s01raw2.png)

**Held-out accuracy: 79.8%** (chance 50 %), rising from 74% for the 20 % least
asymmetric cells (|nucleoid pole asymmetry| < 0.036) to **89%** for the 20 % most
asymmetric. The snapshot itself carries most of the pole identity — the new pole is recognisable
from the nucleoid's lean and the polysome distribution — and tracking is needed only for the
near-symmetric cells. This is the measured motivation for the oriC orientation reference: an
origin-proximal marker in the same snapshot would supply the orientation for every cell.

### 11.3 How the model predicts polarity — the dissection

Every held-out cell is presented *unoriented* (kept or flipped at random); the model's latent is
asked which end is the new pole. The figure takes the prediction apart.

![polarity](figures/figv6_15_polarity_L8s01raw2.png)

**A — the polarity score is a linear function of the pole-signed axes.** A logistic regression on
the three signed axes alone, score = -0.83·z2 -0.93·z3 +0.36·z8,
separates the two orientations at **76.4%**; the forest on all axes reaches
79.6% (AUC 0.883, panel B). Polarity is not hidden in the latent: it
*is* the sign of the pole-signed coordinates. A cell whose score is near zero has no discernible
polarity in its profiles.

**C — polarity is clearest at birth and fades through the cycle**: 89% in the
first decile of the cycle, falling to 73% in the last. A newborn cell carries
the asymmetry it inherited from division — the new pole is the former septum, polysome-rich and
nucleoid-poor — and that asymmetry is erased as the nucleoid replicates and re-centres, and as the
polysomes redistribute toward both poles before the next division.

**D — one-nucleoid cells are easier** (83%) than two-nucleoid cells
(76%): a single nucleoid leaning toward one pole is a clear signal; two
segregated sisters are close to mirror-symmetric.

**E — which axes carry it.** Shuffling z3 (nucleoid pole asymmetry) or z2 (nucleoid position)
costs most; z8 (polysome displacement) and z1 contribute a little; the pole-blind and
concentration axes nothing — as they should.

**F — what polarity looks like.** The mean profiles of the cells the model calls "new pole left"
and "new pole right" are mirror images of each other: the "left" class has its HupA mass displaced
toward the right (old) pole and its RplA excess at the left (new) pole. So the rule the model has
learned is the one lineage tracking encodes: **the new pole is the polysome-rich, nucleoid-poor
end** — the signature of the recent septum.

**G — re-orienting with the model.** The population-mean HupA profile of the unoriented cells is
symmetric by construction (grey); after flipping back the cells the model calls flipped, the mean
profile (dashed) recovers the true-orientation mean (solid) almost exactly. At the population level
the model's orientation is as good as tracking's; at the single-cell level it is right four times
in five, and the score says which cells to trust.

**H — example cells.** Confident correct calls are cells with a clear nucleoid lean; uncertain
ones are nearly symmetric; the confidently *wrong* ones deserve a note — some are cells whose
profiles genuinely point the other way from their lineage label, which can happen when the septum
side was misassigned or when the inherited asymmetry has already reversed. A snapshot-intrinsic
reference such as oriC would resolve these independently of lineage.

The code (`polarity_v6.py`):

In [ ]:
flip = rng.random(N) < 0.5                                  # the hidden truth: was this cell flipped?
Xr = Xc.copy(); Xr[flip] = Xr[flip][:, :, ::-1]            # what the model sees: unoriented cells
Zr = encode(Xr)
Zf = encode(Xc[:, :, ::-1].copy()); equi = np.array([np.corrcoef(Z[:, j], Zf[:, j])[0, 1] for j in range(zdim)])
signed = np.where(equi < 0)[0]

clf = RandomForestClassifier(200, n_jobs=-1, random_state=0, min_samples_leaf=5).fit(Zr[tr], flip[tr])
p = clf.predict_proba(Zr[va])[:, 1]; acc = ((p > 0.5) == flip[va]).mean(); auc = roc_auc_score(flip[va], p)
fpr, tpr, _ = roc_curve(flip[va], p)
# a transparent linear polarity score on the signed axes only
lr = LogisticRegression(max_iter=1000).fit(Zr[tr][:, signed], flip[tr])
score = lr.decision_function(Zr[va][:, signed]); acc_lin = ((score > 0) == flip[va]).mean()
coef = dict(zip([f"z{j+1}" for j in signed], lr.coef_[0]))


* `flip` is the hidden truth; `Xr` is what the model sees; `Zr` its latent; `Zf` (every cell
  flipped) gives the per-axis flip correlation `equi`, and `signed` are the axes that reverse.
* `clf` is the forest on all axes; `p` its P(flipped) on held-out cells; `lr` is the linear score
  on the signed axes only — the transparent version of the same decision.

## 12. Nucleoid position: never an input, recovered

Nucleoid position comes from an independent 2-D segmentation of the HupA image. It is not in the
1-D linescans as such and is never given to the model; it is used only afterwards as a label and a
held-out target.

![position recovery](figures/figv6_11_position_recovery_L8s01raw2.png)

z2 alone correlates with it at ρ = +0.74; the read-out on latent + length recovers it
at R² = 0.699. Without orientation (right panel) the R² collapses to -0.85: the
*magnitude* of the offset is still in the latent but its *sign* is not — section 11 from the other
side.

## 13. Post-processing: pseudo-time, σ-binning, cycle averaging

**13.1 Pseudo-time.** Ordering snapshots along the cycle without tracking, compared by cutting
held-out cells into 25 equal-count bins and taking the SD of the *true* phase inside each bin
(smaller = tighter): cell length 0.181; **z1 alone 0.213** — a cycle-*and*-size axis,
not a pure phase axis, so looser than length; the **phase read-out 0.119**,
34 % tighter than length. The pseudo-time to use is the read-out.

The code (`figures_L8s01.py`):

In [ ]:
def within_bin_sd(key, nb=25):
    edges = np.quantile(key[va], np.linspace(0, 1, nb + 1)); b = np.digitize(key[va], edges[1:-1])
    return np.mean([phase[va][b == k].std() for k in range(nb)])
sd_len, sd_z1 = within_bin_sd(length), within_bin_sd(Z[:, 0])
ro = RandomForestRegressor(100, n_jobs=-1, random_state=0, min_samples_leaf=5).fit(feat[tr], phase[tr])
pt = np.zeros(N); pt[va] = ro.predict(feat[va]); pt[tr] = ro.predict(feat[tr])
sd_pt = within_bin_sd(pt)


* `within_bin_sd(key)`: 25 equal-count bins of `key` on held-out cells; the SD of true phase in
  each, averaged.

![pseudotime](figures/figv6_12_pseudotime_L8s01raw2.png)

### 13.1b Cell length vs the VAE as a cell-cycle clock

Cell length is the conventional snapshot proxy for cycle progression: cells grow, so longer means
later. The proposal's claim is that size-sorting under-resolves the cycle and that a
shape-and-concentration representation does better. This is that claim measured, on held-out
cells, with no tracking used by any method. Five clocks are compared:

| clock | what it uses | held-out R² vs true phase | within-bin SD (25 bins) | distinguishable stages ≈ 1/SD |
|---|---|---|---|---|
| cell length (sorted) | sort by length | 0.602 | 0.181 | 5.6 |
| RF on length only | RF on length | 0.554 | 0.190 | 5.3 |
| RF on scalars (length + concentrations) | RF on length + log RplA + log HupA | 0.739 | 0.147 | 6.8 |
| RF on latent only (VAE) | RF on the 8 latent means | 0.793 | 0.128 | 7.8 |
| RF on latent + length (the read-out) | RF on latent + length | 0.823 | 0.119 | 8.5 |

*Within-bin SD*: cells are cut into 25 equal-count bins along the clock and the SD of their
**true** phase inside each bin is averaged — how blurred a bin is. *Distinguishable stages* is
the inverse of the asymptotic within-bin SD (80 bins): roughly how many cycle stages the clock
can tell apart at one-SD separation.

![length vs VAE](figures/figv6_14_length_vs_vae_L8s01raw2.png)

**A.** Length against true phase: the relation is monotone but broad, and it saturates — above
≈ 4 µm a cell can be anywhere in the last third of its cycle. Cells are born at different sizes
and grow at different rates, so a given length is reached at different phases by different cells;
that variability is the floor length cannot get below (within-bin SD 0.18, ≈ 5.6 stages).

**B.** The VAE read-out against true phase: linear along the diagonal, narrower everywhere
(0.119, ≈ 8.5 stages — 34 % tighter than length). The
compression at both ends is the label's own ambiguity: the frames just after birth and just
before division look alike.

**C.** Resolution against the number of bins. Every curve flattens by ≈ 20 bins: beyond that,
finer bins do not separate cells any better — the asymptote is the clock's intrinsic blur. Length
flattens at 0.18; the concentrations alone (no VAE) already improve it to 0.15, because the
ribosome/DNA ratio tracks growth state and so the cycle; the latent alone reaches 0.13; latent +
length 0.12. The d′ = 1 line (bin width = within-bin SD) is crossed by every clock at ≈ 5 bins:
**no snapshot clock resolves the cycle into more than ~5 fully separated stages** — what the VAE
changes is how much each stage is blurred, not that separation limit.

**D.** Where the clocks fail. Length is sharpest at the two ends of the cycle (newborn cells are
short, dividing cells long) and worst in the **middle** (SD 0.22 at phase 0.4–0.5), exactly where
the biology is happening — replication, nucleoid splitting, segregation — and where size is least
informative because growth-rate variability has had time to accumulate. The VAE read-out is
flattest there (0.15): it reads the nucleoid-splitting programme from the HupA profile (z1:
sister-nucleoid distance, nucleoid number) and does not need size to know that a cell is
mid-cycle. Near division both converge.

**What "post-processing" adds.** The latent alone (0.128) already beats length; adding length to
the read-out (0.119) helps at the ends where size is informative; the random-forest calibration is
what turns an ordering into a phase in [0, 1] with a known blur per bin. Cycle-averaging is not
applicable to phase (it varies within a cycle); the counterpart for a catalogue is to report, per
identity, the phase distribution of its cells — panel D is that distribution's width as a function
of where in the cycle the identity sits.

The code (`length_vs_vae.py`):

In [ ]:
def rf_pred(Xf):
    m = RandomForestRegressor(100, n_jobs=-1, random_state=0, min_samples_leaf=5).fit(Xf[tr], phase[tr])
    return m.predict(Xf[va])

KEYS = {"cell length (sorted)": length[va],
        "RF on length only": rf_pred(length[:, None]),
        "RF on scalars (length + concentrations)": rf_pred(np.c_[length, log_r, log_h]),
        "RF on latent only (VAE)": rf_pred(Z),
        "RF on latent + length (the read-out)": rf_pred(np.c_[Z, length])}


In [ ]:
def binned_sd(key, nb):
    edges = np.quantile(key, np.linspace(0, 1, nb + 1)); b = np.clip(np.digitize(key, edges[1:-1]), 0, nb - 1)
    sds = np.array([pv[b == k].std() for k in range(nb)]); means = np.array([pv[b == k].mean() for k in range(nb)])
    return sds, means, b
NB = [5, 10, 15, 20, 25, 30, 40, 50, 60, 80]
curves = {k: np.array([binned_sd(v, nb)[0].mean() for nb in NB]) for k, v in KEYS.items()}
# resolution limit (d' = 1): the finest nb at which adjacent bins are separated by at least one
# within-bin SD, i.e. within-bin SD <= bin width 1/nb; and the asymptotic resolution 1/SD(80 bins)
limit, stages = {}, {}
for k in KEYS:
    ok = [nb for nb, sd in zip(NB, curves[k]) if sd <= 1.0 / nb]
    limit[k] = max(ok) if ok else 0
    stages[k] = 1.0 / curves[k][-1]


* `rf_pred`: a forest fitted on the training cells for each input set, predicting phase on the
  held-out cells; sorting by length needs no model.
* `binned_sd(key, nb)`: `nb` equal-count bins along the clock; SD and mean of true phase in each.
* `curves`: the mean within-bin SD for 5 … 80 bins; `limit`: the finest binning at which adjacent
  bins are one SD apart (d′ = 1); `stages`: 1 / SD at 80 bins, the asymptotic resolution.

**13.2 σ-binning and cell identities.** Bin width = 2 × median posterior σ over the p1–p99 span
gives **{' / '.join(str(int(b)) for b in bins)}** resolvable bins for z1 … z8. Binning all eight axes
at that resolution gives nearly every cell its own identity — the regime of the planned ~1.7 M-cell
campaign. With the current data a catalogue chooses its axes, e.g. z1 at 40 bins × terciles of the
position, compaction and growth-law axes.

In [ ]:
import numpy as np
n = np.load("latents_L8s01raw2.npz"); order = np.argsort(-n["Z"].std(0)); Z, SIG = n["Z"][:, order], n["SIG"][:, order]
span = np.percentile(Z, 99, 0) - np.percentile(Z, 1, 0)
bins = np.maximum(1, np.floor(span / (2 * np.median(SIG, 0)))).astype(int)
print("resolvable bins per axis:", bins)
# a growth-aware catalogue: z1 at 40 bins x terciles of z2 (position), z4 (compaction), z5 (growth law)
cols = [np.digitize(Z[:, 0], np.linspace(*np.percentile(Z[:, 0], [1, 99]), 41)[1:-1])]
cols += [np.digitize(Z[:, j], np.quantile(Z[:, j], [1/3, 2/3])) for j in (1, 3, 4)]
_, counts = np.unique(np.stack(cols, 1), axis=0, return_counts=True)
print(f"identities {len(counts):,}  median cells {np.median(counts):.0f}  "
      f"share of cells in >=100-cell identities {counts[counts >= 100].sum() / counts.sum():.1%}")

**13.3 Cycle averaging.** When a cell's frames are known to belong together, its latent is
averaged over the cycle before the read-out; this lifts growth R² from 0.51 to 0.58.
For true snapshots the counterpart is averaging over the cells of one identity.

## 14. L8s01raw2 vs L8s01: what the decomposition did and did not buy

Same network, scalars, σ_s, β, schedule, split and seed; only the profile channels differ.

| latent-only read-out, held-out R² | phase | growth frame | growth cycle | position | compaction | polysome asym. | ratio from latent |
|---|---|---|---|---|---|---|---|
| L8s01 (sym/anti, 4 ch) | 0.778 | 0.491 | 0.559 | 0.695 | 0.949 | **0.831** | 0.827 |
| **L8s01raw2 (raw, 2 ch)** | **0.793** | **0.513** | 0.550 | **0.704** | 0.952 | 0.792 | **0.941** |

| | scalars held (RMSE, SD units) | signed axes: flip correlation | concentration basis | strongest growth axis |
|---|---|---|---|---|
| L8s01 | 0.32 / 0.40 / 0.39 | −0.34, −0.45, −0.69 | ratio (z7) + brightness (z8) | z7, ρ −0.45 |
| L8s01raw2 | 0.262 / 0.228 / 0.220 | -0.665, -0.735, -0.607 | log HupA (z5) + log RplA (z6) | z5, ρ +0.52 |

![comparison](figures/figv6_13_raw2.png)

1. **Information: equal.** The decomposition is an invertible linear map; the network receives the
   same content either way. The read-out numbers differ by 0.01–0.04, within single-seed noise,
   and lean toward the raw model on all targets but polysome asymmetry.
2. **Symmetry structure: the raw model finds it unaided, and more cleanly.** The decomposition was
   an interpretability device; what it made explicit, the network learns.
3. **Concentrations: held tighter by the raw model** (two profile channels compete less with the
   scalars), with a per-channel basis in the concentration plane instead of ratio/sum — a rotation
   of the same plane, seed-dependent, with no effect on what the read-out can recover.
4. **Simplicity.** Two channels, no preprocessing beyond mean-normalisation and orientation. The
   symmetry labels are still available — as measurements on the trained model.

The raw two-channel input is therefore the production design; the sym/anti representation remains
the right tool for the v4 dissection (it is what made the mirroring effect measurable).

## 15. Caveats, settings, files

* **Single seed.** Every number is from one training run. Differences below ≈ 0.02 R² between
  models are within seed variation; the concentration *plane* and the growth axis are robust
  findings, the basis within the plane (per-channel vs ratio/sum) is not expected to be.
* **σ_s = 0.1 is a choice**, to be stated wherever the model is described. 0.3 loses the
  concentrations; 0.05 would hold them harder at a further small cost to shape.
* **Session dependence.** With intensities inside the latent, the representation is as portable as
  the intensity calibration; per-session excitation factors are a required pipeline step.
* **Width.** Eight dimensions is the reference width; the 4–5-dimensional catalogue version of this
  design is the next experiment, decided by the binning-occupancy argument of v4.

**Settings at a glance.** 2 centred profile channels × 100 bins + 3 standardised scalars; 8 latent
dims; β = 2; σ_x learned per channel; σ_s = 0.1 fixed; Adam 10⁻³; batch 512; 40 epochs; seed 0;
split by cycle (159,835 / 27,846). Read-out: RandomForestRegressor(100 trees, min leaf 5) on
[μ, length]; cycle-averaged growth: 200 trees on per-cycle means.

**Files.**

| file | role |
|---|---|
| `train_vae_v6.py 0.1 --raw2` | training |
| `../v4/vae_model_v4.py` | network, `prep_profiles()`, `reassemble()`, `load_model()`, `scalar_inputs()` |
| `vae_L8s01raw2.pt`, `latents_L8s01raw2.npz` | checkpoint; posterior μ and σ for all cells |
| `evaluate_v6.py`, `results_v6.md` | read-outs Z/B/A/C, five models, linear probes |
| `inspect_v6.py L8s01raw2`, `inspect_log_raw2.txt` | purity, flip equivariance, scalar-head response, fig 1 |
| `figures_L8s01.py L8s01raw2` | figs 2–12 (data, inputs, training, latent, growth law, population, traversals, read-out, pole, position, post-processing) |
| `compare_raw2.py` | the side-by-side with L8s01 (fig 13) |
| `length_vs_vae.py L8s01raw2` | cell length vs the VAE as a cycle clock (fig 14) |
| `polarity_v6.py L8s01raw2` | how polarity is predicted: score, ROC, by phase / nucleoid number, axes, mean profiles, examples (fig 15) |
| `../v4/schematic_v4.py L8s01raw2 ../v6` | fig 0 |
| `../v4/figures_standard_v4.py`, `figures_atlas_v4.py`, `unoriented_correlations_v4.py`, `make_projections_v4.py` (each `L8s01raw2 ../v6`) | reconstructions, traversals, atlas, σ-binning, unoriented correlations |
| `build_notebook_raw2.py` | builds this notebook |

Data: `../v2/dataset_v2.npz` (187,681 frames, 4,122 cycles; BioImage Archive **S-BIAD1658**).